In [3]:
# # survey_classwise_stats.py
# import os
# import numpy as np
# import pandas as pd
# import matplotlib.pyplot as plt
# from collections import defaultdict

# # ===================== CẤU HÌNH =====================
# TRAIN_CSV = "Paths_and_Labels_of_TrainingSet.csv"
# TEST_CSV  = "Paths_and_Labels_of_250915_TestingSet.csv"

# # Bật/tắt vẽ hình
# PLOT_PER_CLASS_HIST = False
# PLOT_PER_CLASS_BOXPLOT = False
# PLOT_GLOBAL_HIST = False

# # Lưu CSV kết quả vào thư mục artifacts
# SAVE_CSV = True
# ARTIFACT_DIR = "artifacts_survey"

# # (Tuỳ chọn) Chuẩn hoá mỗi mẫu trước khi thống kê (z-score theo từng file)
# PER_SAMPLE_STANDARDIZE = False  # True nếu bạn muốn loại ảnh hưởng biên độ giữa mẫu
# # ====================================================


# def ensure_dir(path):
#     os.makedirs(path, exist_ok=True)


# def load_csv(csv_path):
#     """Đọc file CSV, trả về (paths, labels_raw). Yêu cầu cột 'Paths' và 'True labels'."""
#     df = pd.read_csv(csv_path)
#     if not {"Paths", "True labels"}.issubset(df.columns):
#         raise ValueError(f"{csv_path} phải có cột 'Paths' và 'True labels'. Cột hiện có: {list(df.columns)}")
#     x_paths = df["Paths"].values
#     y_raw = df["True labels"].values
#     return x_paths, y_raw


# def normalize_sample(arr):
#     """Chuẩn hoá theo từng mẫu: (x - mean)/std, an toàn với std=0."""
#     mu = arr.mean()
#     sigma = arr.std()
#     if sigma < 1e-12:
#         return arr - mu
#     return (arr - mu) / sigma


# def load_flatten_values(paths):
#     """
#     Đọc toàn bộ file trong danh sách paths.
#     Trả về list các vector 1D (mỗi phần tử là 1 sample flattened).
#     """
#     flat_list = []
#     for p in paths:
#         arr = np.loadtxt(p, delimiter=",").astype(np.float32)
#         if PER_SAMPLE_STANDARDIZE:
#             arr = normalize_sample(arr)
#         flat_list.append(arr.flatten())
#     return flat_list


# def classwise_stats(x_paths, y_labels, class_names, set_name="Train", plot_hist=False, plot_box=False):
#     """
#     Tính thống kê theo từng class.
#     y_labels có thể là chuỗi hoặc số. class_names là danh sách tên lớp chuẩn hoá để in ra.
#     Trả về DataFrame thống kê và dict {class_name: list_of_flat_values}
#     """
#     # gom paths theo class index (dựa trên class_names)
#     # tạo mapping label->class_index
#     # Trường hợp y_labels là chuỗi: map theo tên, nếu là số: map theo thứ tự unique đã tương ứng class_names
#     label_to_index = {}
#     # Nếu y_labels là chuỗi, class_names cũng chuỗi → map trực tiếp
#     if isinstance(class_names[0], str) and isinstance(y_labels[0], str):
#         classes_set = set(class_names)
#         for i, cname in enumerate(class_names):
#             label_to_index[cname] = i
#         class_indices = np.array([label_to_index.get(lbl, -1) for lbl in y_labels])
#     else:
#         # Trường hợp y_labels là số (đã mã hoá), giả định 0..K-1 tương ứng class_names
#         class_indices = np.array(y_labels, dtype=int)

#     # gom paths theo class
#     grouped_paths = defaultdict(list)
#     for i, idx in enumerate(class_indices):
#         if idx < 0 or idx >= len(class_names):
#             continue
#         grouped_paths[idx].append(x_paths[i])

#     # tính thống kê
#     records = []
#     class_values_dict = {}
#     for idx, cname in enumerate(class_names):
#         paths = grouped_paths.get(idx, [])
#         if len(paths) == 0:
#             # không có mẫu cho class này
#             records.append({
#                 "Class": cname, "Count": 0, "ValuesCount": 0,
#                 "Mean": np.nan, "Std": np.nan, "Min": np.nan,
#                 "25%": np.nan, "Median": np.nan, "75%": np.nan,
#                 "Max": np.nan, "IQR": np.nan,
#             })
#             class_values_dict[cname] = []
#             continue

#         flat_list = load_flatten_values(paths)
#         # ghép tất cả values của class
#         all_values = np.concatenate(flat_list, axis=0)

#         stats = {
#             "Class": cname,
#             "Count": len(paths),
#             "ValuesCount": all_values.size,
#             "Mean": float(np.mean(all_values)),
#             "Std": float(np.std(all_values)),
#             "Min": float(np.min(all_values)),
#             "25%": float(np.percentile(all_values, 25)),
#             "Median": float(np.median(all_values)),
#             "75%": float(np.percentile(all_values, 75)),
#             "Max": float(np.max(all_values)),
#         }
#         stats["IQR"] = stats["75%"] - stats["25%"]
#         records.append(stats)
#         class_values_dict[cname] = all_values  # lưu lại để vẽ biểu đồ

#         # vẽ histogram từng class (tuỳ chọn)
#         if plot_hist:
#             plt.figure(figsize=(7, 3))
#             plt.hist(all_values, bins=80)
#             plt.title(f"{set_name} - Class: {cname} (n={len(paths)})")
#             plt.xlabel("Value")
#             plt.ylabel("Count")
#             plt.grid(True)
#             plt.show()

#     df = pd.DataFrame.from_records(records)
#     # vẽ boxplot theo class (tuỳ chọn)
#     if plot_box:
#         # tạo dataframe dài cho boxplot
#         long_records = []
#         for cname, vals in class_values_dict.items():
#             if len(vals) == 0:
#                 continue
#             # downsample nếu quá lớn để plot nhanh hơn
#             if vals.size > 50000:
#                 # lấy ngẫu nhiên 50k điểm để vẽ
#                 sel = np.random.choice(vals.size, 50000, replace=False)
#                 vals = vals[sel]
#             long_records.append(pd.DataFrame({"Class": cname, "Value": vals}))
#         if long_records:
#             df_long = pd.concat(long_records, axis=0, ignore_index=True)
#             plt.figure(figsize=(10, 4))
#             df_long.boxplot(by="Class", column=["Value"], grid=True, rot=30)
#             plt.suptitle("")
#             plt.title(f"{set_name} - Boxplot per Class")
#             plt.xlabel("Class")
#             plt.ylabel("Value")
#             plt.tight_layout()
#             plt.show()

#     return df, class_values_dict


# def guess_class_names(y_train_raw, y_test_raw):
#     """
#     Xác định danh sách class_names theo dữ liệu đầu vào:
#     - Nếu nhãn là chuỗi: lấy union (giữ thứ tự theo sắp xếp alpha).
#     - Nếu nhãn là số: giả định 0..K-1; tạo tên 'Class0', 'Class1', ...
#       Bạn có thể sửa lại tên theo hệ của bạn nếu muốn.
#     """
#     y_all = np.concatenate([y_train_raw, y_test_raw], axis=0)
#     if isinstance(y_all[0], str):
#         classes = sorted(list(set(y_all.tolist())))
#         return classes
#     else:
#         uniq = sorted(list(set(y_all.tolist())))
#         return [f"Class{u}" for u in uniq]


# def global_histogram(values_per_set, set_names):
#     """Vẽ histogram toàn tập cho Train/Test (tuỳ chọn)."""
#     for vals, name in zip(values_per_set, set_names):
#         if vals.size == 0:
#             continue
#         plt.figure(figsize=(8, 4))
#         plt.hist(vals, bins=120)
#         plt.title(f"Global Value Distribution - {name}")
#         plt.xlabel("Value")
#         plt.ylabel("Count")
#         plt.grid(True)
#         plt.show()


# def compare_train_test(df_train, df_test):
#     """
#     Ghép bảng thống kê Train/Test theo class và thêm cột chênh lệch.
#     """
#     keep_cols = ["Class", "Count", "Mean", "Std"]
#     t = df_train[keep_cols].rename(columns={
#         "Count": "Count_Train", "Mean": "Mean_Train", "Std": "Std_Train"
#     })
#     s = df_test[keep_cols].rename(columns={
#         "Count": "Count_Test", "Mean": "Mean_Test", "Std": "Std_Test"
#     })
#     merged = pd.merge(t, s, on="Class", how="outer")
#     merged["Count_Ratio_Test/Train"] = merged["Count_Test"] / merged["Count_Train"]
#     merged["Delta_Mean"] = merged["Mean_Test"] - merged["Mean_Train"]
#     merged["Delta_Std"] = merged["Std_Test"] - merged["Std_Train"]
#     return merged


# def main():
#     np.random.seed(42)
#     if SAVE_CSV:
#         ensure_dir(ARTIFACT_DIR)

#     # 1) Đọc CSV
#     x_train, y_train_raw = load_csv(TRAIN_CSV)
#     x_test,  y_test_raw  = load_csv(TEST_CSV)

#     # 2) Xác định class_names (tự động)
#     class_names = guess_class_names(y_train_raw, y_test_raw)
#     print("Class names used:", class_names)

#     # 3) Thống kê theo class (Train/Test)
#     df_train, train_values_dict = classwise_stats(
#         x_train, y_train_raw, class_names,
#         set_name="Train",
#         plot_hist=PLOT_PER_CLASS_HIST,
#         plot_box=PLOT_PER_CLASS_BOXPLOT
#     )
#     df_test, test_values_dict = classwise_stats(
#         x_test, y_test_raw, class_names,
#         set_name="Test",
#         plot_hist=PLOT_PER_CLASS_HIST,
#         plot_box=PLOT_PER_CLASS_BOXPLOT
#     )

#     # 4) In & lưu bảng
#     print("\n=== Class-wise Statistics: Train ===")
#     print(df_train)
#     print("\n=== Class-wise Statistics: Test ===")
#     print(df_test)

#     if SAVE_CSV:
#         df_train.to_csv(os.path.join(ARTIFACT_DIR, "train_classwise_stats.csv"), index=False)
#         df_test.to_csv(os.path.join(ARTIFACT_DIR, "test_classwise_stats.csv"), index=False)

#     # 5) So sánh Train vs Test theo class
#     df_cmp = compare_train_test(df_train, df_test)
#     print("\n=== Compare Train vs Test per Class ===")
#     print(df_cmp.sort_values("Class"))

#     if SAVE_CSV:
#         df_cmp.to_csv(os.path.join(ARTIFACT_DIR, "compare_train_test_per_class.csv"), index=False)

#     # 6) (Tuỳ chọn) Vẽ histogram toàn tập
#     if PLOT_GLOBAL_HIST:
#         # ghép toàn bộ values train/test
#         train_all = []
#         for v in train_values_dict.values():
#             if len(v) > 0:
#                 train_all.append(v)
#         test_all = []
#         for v in test_values_dict.values():
#             if len(v) > 0:
#                 test_all.append(v)
#         train_all = np.concatenate(train_all) if train_all else np.array([])
#         test_all  = np.concatenate(test_all)  if test_all  else np.array([])
#         global_histogram([train_all, test_all], ["Train", "Test"])

#     # 7) Cảnh báo nhanh về shift (đơn giản)
#     warn = []
#     for _, row in df_cmp.iterrows():
#         c = row["Class"]
#         # cảnh báo nếu Count lệch nhiều, hoặc Mean/Std lệch lớn
#         ratio = row["Count_Ratio_Test/Train"]
#         dmean = row["Delta_Mean"]
#         dstd  = row["Delta_Std"]
#         if pd.notna(ratio) and (ratio < 0.5 or ratio > 2.0):
#             warn.append(f"- Class {c}: Count ratio Test/Train = {ratio:.2f} (mất cân bằng)")
#         if pd.notna(dmean) and abs(dmean) > 0.25 * (abs(row["Mean_Train"]) + 1e-6):
#             warn.append(f"- Class {c}: |Delta Mean| lớn ({dmean:.4g})")
#         if pd.notna(dstd) and abs(dstd) > 0.25 * (row["Std_Train"] + 1e-6):
#             warn.append(f"- Class {c}: |Delta Std| lớn ({dstd:.4g})")

#     if warn:
#         print("\n⚠️  Cảnh báo khả năng dataset shift:")
#         for w in warn:
#             print(w)
#     else:
#         print("\n✅ Không phát hiện lệch phân bố đáng kể theo tiêu chí đơn giản (có thể tinh chỉnh ngưỡng).")


# if __name__ == "__main__":
#     main()


Class names used: ['Corona', 'Floating', 'Noise', 'Surface', 'Turn_to_Turn', 'Void']

=== Class-wise Statistics: Train ===
          Class  Count  ValuesCount       Mean        Std  Min  25%  Median  \
0        Corona    161     74188800   7.419895   2.176462  2.0  6.0     7.0   
1      Floating     41     18892800  13.004428  38.089058  2.0  6.0     7.0   
2         Noise    317    146073600  12.129668   5.003249  0.0  7.0    14.0   
3       Surface    135     62208000   3.526999   5.035132  0.0  0.0     0.0   
4  Turn_to_Turn     44     20275200   1.789710  11.839670  0.0  0.0     0.0   
5          Void    113     52070400   8.796264  18.199520  0.0  0.0     0.0   

    75%    Max  IQR  
0   8.0  169.0  2.0  
1   8.0  255.0  2.0  
2  16.0  217.0  9.0  
3   8.0   86.0  8.0  
4   0.0  189.0  0.0  
5   7.0  210.0  7.0  

=== Class-wise Statistics: Test ===
          Class  Count  ValuesCount       Mean        Std  Min  25%  Median  \
0        Corona     27     12441600   8.471087   3.45

In [5]:
# survey_classwise_stats_plus.py
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import defaultdict

# ===================== CẤU HÌNH =====================
TRAIN_CSV = "Paths_and_Labels_of_TrainingSet.csv"
TEST_CSV  = "Paths_and_Labels_of_TestingSet.csv"

PLOT_PER_CLASS_HIST = False
PLOT_PER_CLASS_BOXPLOT = False
PLOT_GLOBAL_HIST = False

SAVE_CSV = True
ARTIFACT_DIR = "artifacts_survey"

PER_SAMPLE_STANDARDIZE = False  # giữ nguyên dữ liệu (False)
# ====================================================


def ensure_dir(path):
    os.makedirs(path, exist_ok=True)


def load_csv(csv_path):
    df = pd.read_csv(csv_path)
    if not {"Paths", "True labels"}.issubset(df.columns):
        raise ValueError(f"{csv_path} phải có cột 'Paths' và 'True labels'. Cột hiện có: {list(df.columns)}")
    return df["Paths"].values, df["True labels"].values


def normalize_sample(arr):
    mu = arr.mean()
    sigma = arr.std()
    if sigma < 1e-12:
        return arr - mu
    return (arr - mu) / sigma


def load_flatten_values(paths):
    flat_list = []
    for p in paths:
        arr = np.loadtxt(p, delimiter=",").astype(np.float32)
        if PER_SAMPLE_STANDARDIZE:
            arr = normalize_sample(arr)
        flat_list.append(arr.flatten())
    return flat_list


def classwise_stats(x_paths, y_labels, class_names, set_name="Train", plot_hist=False, plot_box=False):
    if isinstance(class_names[0], str) and isinstance(y_labels[0], str):
        label_to_index = {c: i for i, c in enumerate(class_names)}
        class_indices = np.array([label_to_index.get(lbl, -1) for lbl in y_labels])
    else:
        class_indices = np.array(y_labels, dtype=int)

    grouped_paths = defaultdict(list)
    for i, idx in enumerate(class_indices):
        if 0 <= idx < len(class_names):
            grouped_paths[idx].append(x_paths[i])

    records = []
    class_values_dict = {}
    for idx, cname in enumerate(class_names):
        paths = grouped_paths.get(idx, [])
        if len(paths) == 0:
            records.append({
                "Class": cname, "Count": 0, "ValuesCount": 0,
                "Mean": np.nan, "Std": np.nan,
                "Mean_of_Max": np.nan, "Std_of_Mean": np.nan,
                "Min": np.nan, "25%": np.nan, "Median": np.nan,
                "75%": np.nan, "Max": np.nan, "IQR": np.nan,
            })
            class_values_dict[cname] = []
            continue

        flat_list = []
        per_file_mean = []
        per_file_max = []
        for p in paths:
            arr = np.loadtxt(p, delimiter=",").astype(np.float32)
            if PER_SAMPLE_STANDARDIZE:
                arr = normalize_sample(arr)
            flat_list.append(arr.flatten())
            per_file_mean.append(arr.mean())
            per_file_max.append(arr.max())

        all_values = np.concatenate(flat_list, axis=0)

        stats = {
            "Class": cname,
            "Count": len(paths),
            "ValuesCount": all_values.size,
            "Mean": float(np.mean(all_values)),
            "Std": float(np.std(all_values)),
            "Mean_of_Max": float(np.mean(per_file_max)),  # ✅ trung bình max
            "Std_of_Mean": float(np.std(per_file_mean)),  # ✅ độ lệch chuẩn mean
            "Min": float(np.min(all_values)),
            "25%": float(np.percentile(all_values, 25)),
            "Median": float(np.median(all_values)),
            "75%": float(np.percentile(all_values, 75)),
            "Max": float(np.max(all_values)),
        }
        stats["IQR"] = stats["75%"] - stats["25%"]
        records.append(stats)
        class_values_dict[cname] = all_values

        if plot_hist:
            plt.figure(figsize=(7, 3))
            plt.hist(all_values, bins=80)
            plt.title(f"{set_name} - Class: {cname} (n={len(paths)})")
            plt.xlabel("Value")
            plt.ylabel("Count")
            plt.grid(True)
            plt.show()

    df = pd.DataFrame.from_records(records)

    if plot_box:
        long_records = []
        for cname, vals in class_values_dict.items():
            if len(vals) == 0:
                continue
            if vals.size > 50000:
                sel = np.random.choice(vals.size, 50000, replace=False)
                vals = vals[sel]
            long_records.append(pd.DataFrame({"Class": cname, "Value": vals}))
        if long_records:
            df_long = pd.concat(long_records, axis=0, ignore_index=True)
            plt.figure(figsize=(10, 4))
            df_long.boxplot(by="Class", column=["Value"], grid=True, rot=30)
            plt.suptitle("")
            plt.title(f"{set_name} - Boxplot per Class")
            plt.xlabel("Class")
            plt.ylabel("Value")
            plt.tight_layout()
            plt.show()

    return df, class_values_dict


def guess_class_names(y_train_raw, y_test_raw):
    y_all = np.concatenate([y_train_raw, y_test_raw], axis=0)
    if isinstance(y_all[0], str):
        return sorted(list(set(y_all.tolist())))
    else:
        uniq = sorted(list(set(y_all.tolist())))
        return [f"Class{u}" for u in uniq]


def global_histogram(values_per_set, set_names):
    for vals, name in zip(values_per_set, set_names):
        if vals.size == 0:
            continue
        plt.figure(figsize=(8, 4))
        plt.hist(vals, bins=120)
        plt.title(f"Global Value Distribution - {name}")
        plt.xlabel("Value")
        plt.ylabel("Count")
        plt.grid(True)
        plt.show()


def compare_train_test(df_train, df_test):
    keep_cols = ["Class", "Count", "Mean", "Std", "Mean_of_Max", "Std_of_Mean"]
    t = df_train[keep_cols].rename(columns={c: c+"_Train" for c in keep_cols if c != "Class"})
    s = df_test[keep_cols].rename(columns={c: c+"_Test" for c in keep_cols if c != "Class"})
    merged = pd.merge(t, s, on="Class", how="outer")
    merged["Count_Ratio_Test/Train"] = merged["Count_Test"] / merged["Count_Train"]
    merged["Delta_Mean"] = merged["Mean_Test"] - merged["Mean_Train"]
    merged["Delta_Std"] = merged["Std_Test"] - merged["Std_Train"]
    return merged


def main():
    np.random.seed(42)
    if SAVE_CSV:
        ensure_dir(ARTIFACT_DIR)

    x_train, y_train_raw = load_csv(TRAIN_CSV)
    x_test, y_test_raw = load_csv(TEST_CSV)

    class_names = guess_class_names(y_train_raw, y_test_raw)
    print("Class names used:", class_names)

    df_train, train_values_dict = classwise_stats(x_train, y_train_raw, class_names,
                                                 set_name="Train", plot_hist=PLOT_PER_CLASS_HIST)
    df_test, test_values_dict = classwise_stats(x_test, y_test_raw, class_names,
                                               set_name="Test", plot_hist=PLOT_PER_CLASS_HIST)

    print("\n=== Class-wise Statistics: Train ===")
    print(df_train)
    print("\n=== Class-wise Statistics: Test ===")
    print(df_test)

    if SAVE_CSV:
        df_train.to_csv(os.path.join(ARTIFACT_DIR, "train_classwise_stats_plus.csv"), index=False)
        df_test.to_csv(os.path.join(ARTIFACT_DIR, "test_classwise_stats_plus.csv"), index=False)

    df_cmp = compare_train_test(df_train, df_test)
    print("\n=== Compare Train vs Test per Class ===")
    print(df_cmp.sort_values("Class"))

    if SAVE_CSV:
        df_cmp.to_csv(os.path.join(ARTIFACT_DIR, "compare_train_test_per_class_plus.csv"), index=False)


if __name__ == "__main__":
    main()


Class names used: ['Corona', 'Floating', 'Noise', 'Surface', 'Turn_to_Turn', 'Void']

=== Class-wise Statistics: Train ===
          Class  Count  ValuesCount       Mean        Std  Mean_of_Max  \
0        Corona    161     74188800   7.419895   2.176462   128.341614   
1      Floating     41     18892800  13.004428  38.089058   255.000000   
2         Noise    317    146073600  12.129668   5.003249    26.268139   
3       Surface    135     62208000   3.526999   5.035132    59.733334   
4  Turn_to_Turn     44     20275200   1.789710  11.839670   139.545456   
5          Void    113     52070400   8.796264  18.199520   110.230087   

   Std_of_Mean  Min  25%  Median   75%    Max  IQR  
0     0.314910  2.0  6.0     7.0   8.0  169.0  2.0  
1     1.941637  2.0  6.0     7.0   8.0  255.0  2.0  
2     4.636124  0.0  7.0    14.0  16.0  217.0  9.0  
3     3.976764  0.0  0.0     0.0   8.0   86.0  8.0  
4     1.703020  0.0  0.0     0.0   0.0  189.0  0.0  
5     2.845258  0.0  0.0     0.0   7.0  